In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [2]:
import os
os.chdir(r'C:\Users\AnuragS\OneDrive\cv_project\Movie Recommendation System\data')
os.getcwd()

'C:\\Users\\AnuragS\\OneDrive\\cv_project\\Movie Recommendation System\\data'

In [3]:
df = pd.read_csv('tmdb_movie_merged.csv')
df.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [4]:
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'cast', 'crew'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [6]:
drop_cols = [
    'budget', ## not important for recommendation
    'homepage', ## 
    'original_language', ## among 5k movie 4505 are English movies (imbalanced)
    'original_title', ## language may differ, title column will be taken instead
    'popularity',
    'production_companies', ## generally movies aren't recommended based on production companies
    'production_countries',
    'revenue', ## 
    'runtime',
    'spoken_languages',
    'status',
    'tagline', ## will not help
    'vote_count',
    'vote_average'
]

In [7]:
req_col = [col for col in df.columns if col not in drop_cols]
df = df[req_col]

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   genres        4803 non-null   str  
 1   id            4803 non-null   int64
 2   keywords      4803 non-null   str  
 3   overview      4800 non-null   str  
 4   release_date  4802 non-null   str  
 5   title         4803 non-null   str  
 6   cast          4803 non-null   str  
 7   crew          4803 non-null   str  
dtypes: int64(1), str(7)
memory usage: 300.3 KB


In [9]:
df['release_date'] = pd.to_datetime(df['release_date'])
df['release_year'] = df['release_date'].dt.year

In [10]:
df.head(2)

,genres,id,keywords,overview,release_date,title,cast,crew,release_year
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",2009-12-10,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2009.0
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha...",2007-05-19,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2007.0


In [11]:
df.isnull().sum()

genres          0
id              0
keywords        0
overview        3
release_date    1
title           0
cast            0
crew            0
release_year    1
dtype: int64

In [12]:
df.dropna(inplace=True)

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df['genres'] = df['genres'].apply(ast.literal_eval).apply(lambda x:[g['name'] for g in x])
df['keywords'] = df['keywords'].apply(ast.literal_eval).apply(lambda x:[g['name'] for g in x])
df['crew'] = df['crew'].apply(ast.literal_eval).apply(lambda x: [g['name'] for g in x if g['job'] == 'Director'])
df['cast'] = df['cast'].apply(ast.literal_eval).apply(lambda x: [g['name'] for g in x[:3]])
df['overview'] = df['overview'].apply(lambda x: x.split())

In [15]:
df.head(2)

,genres,id,keywords,overview,release_date,title,cast,crew,release_year
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",2009-12-10,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],2009.0
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...","[Captain, Barbossa,, long, believed, to, be, d...",2007-05-19,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],2007.0


In [16]:
revamped_cols = ['genres','keywords','crew','cast']

for col in revamped_cols:
    df[col] = df[col].apply(lambda x:[i.replace(' ','') for i in x])

In [17]:
df['release_year'] = df['release_year'].astype(int).apply(lambda x:[x])

In [18]:
df.head(3)

,genres,id,keywords,overview,release_date,title,cast,crew,release_year
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",2009-12-10,Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],[2009]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drugabuse, exoticisland, eastindiatrad...","[Captain, Barbossa,, long, believed, to, be, d...",2007-05-19,Pirates of the Caribbean: At World's End,"[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],[2007]
2,"[Action, Adventure, Crime]",206647,"[spy, basedonnovel, secretagent, sequel, mi6, ...","[A, cryptic, message, from, Bond’s, past, send...",2015-10-26,Spectre,"[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],[2015]


In [19]:
df['tag'] = (df['overview']
             + df['genres']
             + df['overview']
             + df['keywords']
             + df['cast']
             + df['crew']
             + df['release_year']
)

In [20]:
df_ana = df[['id','title','tag']]
df_ana

,id,title,tag
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4798,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4799,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4800,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4801,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [21]:
df_ana['tag'] = df_ana['tag'].apply(lambda x:' '.join(map(str,x))).apply(lambda y:y.lower())


In [22]:
df_ana.head()

,id,title,tag
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [23]:
df.to_csv('cleaned_data.csv',index=False)
df_ana.to_csv('cleaned_data_for_anylytics.csv',index=False)